# AI Question Answering System - EDA & Model Training

This notebook contains the complete pipeline for exploring the QA dataset and training the TF-IDF retrieval model. It includes data loading, exploratory data analysis (EDA), text preprocessing, model training, and saving the artifacts.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import nltk
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer
from nltk.tokenize import word_tokenize
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
import pickle
import os

import warnings
warnings.filterwarnings('ignore')

## 1. Download NLTK Dependencies
We need tokenizers and lemmatizers for text preprocessing.

In [ ]:
print("Downloading NLTK resources...")
nltk.download('punkt')
nltk.download('stopwords')
nltk.download('wordnet')
nltk.download('omw-1.4')
nltk.download('punkt_tab')

## 2. Load the Dataset
We load the `qa_dataset_clean.csv` which contains over 600,000 QA pairs.

In [ ]:
INPUT_FILE = "dataset/qa_dataset_clean.csv"
df = pd.read_csv(INPUT_FILE)

print(f"Total dataset records: {len(df):,}")
df.head()

## 3. Exploratory Data Analysis (EDA)
Let's analyze the length of the questions and answers to understand the distribution of our text data.

In [ ]:
# Calculate lengths
df['question_length'] = df['question'].astype(str).apply(lambda x: len(x.split()))
df['answer_length'] = df['answer'].astype(str).apply(lambda x: len(x.split()))

print("Question Length Stats:\n", df['question_length'].describe())
print("\nAnswer Length Stats:\n", df['answer_length'].describe())

In [ ]:
plt.figure(figsize=(12, 5))

plt.subplot(1, 2, 1)
sns.histplot(df['question_length'], bins=50, kde=True, color='blue')
plt.title('Distribution of Question Lengths (Words)')
plt.xlabel('Number of Words')
plt.ylabel('Frequency')
plt.xlim(0, 30) # Truncate long tails for better visualization

plt.subplot(1, 2, 2)
sns.histplot(df['answer_length'], bins=50, kde=True, color='green')
plt.title('Distribution of Answer Lengths (Words)')
plt.xlabel('Number of Words')
plt.ylabel('Frequency')
plt.xlim(0, 150)

plt.tight_layout()
plt.show()

## 4. Text Preprocessing
We will clean the questions by lowercasing, tokenizing, removing stopwords, and lemmatizing.

In [ ]:
stop_words = set(stopwords.words('english'))
lemmatizer = WordNetLemmatizer()

def preprocess_text(text):
    if not isinstance(text, str):
        return ""
    tokens = word_tokenize(text.lower())
    cleaned_tokens = [lemmatizer.lemmatize(t) for t in tokens if t.isalpha() and t not in stop_words]
    return " ".join(cleaned_tokens)

print("Sample original question:", df['question'].iloc[0])
print("Sample preprocessed:", preprocess_text(df['question'].iloc[0]))

In [ ]:
# Apply preprocessing to all questions
print("Applying preprocessing to the entire dataset...")
df['processed_question'] = df['question'].apply(preprocess_text)
df[['question', 'processed_question']].head()

## 5. Model Training (TF-IDF Vectorization & Logistic Regression)
We split the dataset into 80/20 train/test segments. Then, we train a TF-IDF vectorizer and a Logistic Regression classification model on the preprocessed questions.

In [ ]:
print("Sampling 10,000 records to prevent local memory exhaustion during multi-class regression...")
df_sample = df.sample(n=min(10000, len(df)), random_state=42).reset_index(drop=True)
print("\nSplitting dataset into 80% training and 20% testing...")
X_train_text, X_test_text, y_train, y_test = train_test_split(
    df_sample['processed_question'], df_sample['answer'], test_size=0.20, random_state=42
)

print("\nFitting TF-IDF Vectorizer on training data...")
vectorizer = TfidfVectorizer(
    ngram_range=(1, 2),
    max_features=10000,
    stop_words="english"
)

X_train = vectorizer.fit_transform(X_train_text)
X_test = vectorizer.transform(X_test_text)
print(f"Vocabulary size: {len(vectorizer.vocabulary_):,}")
print(f"Feature matrix shape: {X_train.shape}")

print("\nTraining Logistic Regression Classifier...")
model = LogisticRegression(max_iter=1000)
model.fit(X_train, y_train)

print("\nEvaluating Model Performance...")
y_pred = model.predict(X_test)
acc = accuracy_score(y_test, y_pred)
prec = precision_score(y_test, y_pred, average='weighted', zero_division=0)
rec = recall_score(y_test, y_pred, average='weighted', zero_division=0)
f1 = f1_score(y_test, y_pred, average='weighted', zero_division=0)
print(f"Accuracy:  {acc:.2f}")
print(f"Precision: {prec:.2f}")
print(f"Recall:    {rec:.2f}")
print(f"F1 Score:  {f1:.2f}")

## 6. Save Model Artifacts
Finally, we save the trained vectorizer and the encoded dataset into `.pkl` files.

In [ ]:
MODEL_DIR = "model"
os.makedirs(MODEL_DIR, exist_ok=True)

vectorizer_path = os.path.join(MODEL_DIR, "vectorizer.pkl")
model_path = os.path.join(MODEL_DIR, "qa_model.pkl")

print("Saving vectorizer...")
with open(vectorizer_path, 'wb') as f:
    pickle.dump(vectorizer, f)

print("Saving ML model artifacts...")
with open(model_path, 'wb') as f:
    pickle.dump(model, f)

print("Training pipeline completed and artifacts saved successfully!")